# Chapter 8 — Task Decomposition and Orchestration · Practice

**Goal:** build every decomposition shape from the chapter with the Agents SDK (pipelines with gates, routing, bounded fan-out, orchestrator-workers, evaluator loops), see the one-handoff-per-turn rule and hosted multi-agent items for yourself, and finish with a PR review desk that plans, fans out, integrates and independently reviews a pull request.

- Theory: [README.md](README.md) (sections 8.1–8.10). This notebook holds the code.
- **Estimated time:** 3–4 hours
- **Estimated API cost:** about $0.003 with the default `gpt-6-luna` if you set `RUN_LIVE_CELLS = True` (it enables the few cells marked **(live)**), plus about $0.02 if you flip `RUN_LIVE` in the mini-project (it also uses `gpt-6.1-sol`). Everything else runs **offline** on a scripted model and costs nothing.
- **Prerequisites:** [Chapter 3](../03-agents-sdk-and-agents-api/README.md) (Agents SDK) and [Chapter 6](../06-prompt-engineering-and-evals/README.md) (chaining, evals), the repo venv from [00-prerequisites](../../00-prerequisites/README.md), and `OPENAI_API_KEY` in `.env` for the live cells.

> Jupyter already runs an event loop, so this notebook uses top-level `await Runner.run(...)` and `await asyncio.gather(...)`. In a script, wrap the same code in `async def main()` and call `asyncio.run(main())`.

In [ ]:
import asyncio
import inspect
import itertools
import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, Field
from agents import (
    Agent,
    AgentOutputSchema,
    ModelBehaviorError,
    ModelSettings,
    RunConfig,
    Runner,
    ToolExecutionConfig,
    TracingProcessor,
    Usage,
    add_trace_processor,
    handoff,
    set_trace_processors,
    set_tracing_disabled,
    trace,
)
from agents.testing import ModelCall, ModelStep, ScriptedModel, assistant_message, function_call

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")  # cheap tier: workers, routers, drafts
STRONG_MODEL = "gpt-6.1-sol"                       # planning, integration review, judging

# A real key means live cells can run and traces can go to the OpenAI Traces dashboard.
# (NBCHECK is set by this repo's notebook checker, which injects a fake key.)
HAVE_REAL_KEY = bool(os.getenv("OPENAI_API_KEY")) and not os.getenv("NBCHECK")
set_tracing_disabled(not HAVE_REAL_KEY)
print(f"MODEL={MODEL} | STRONG_MODEL={STRONG_MODEL} | live API available: {HAVE_REAL_KEY}")

In [ ]:
# Live cells are OPT-IN. Set RUN_LIVE_CELLS = True to run the cells marked (live) (~$0.003 on MODEL).
# With the default False, "Run All" stays offline and costs nothing, with or without a key.
RUN_LIVE_CELLS = False
if RUN_LIVE_CELLS:
    assert HAVE_REAL_KEY, "Set OPENAI_API_KEY in .env (see 00-prerequisites)"

### Offline helper: scripted models

`openai-agents` ships deterministic test doubles in `agents.testing` (README [8.10](README.md#810-verification-contracts-testing-and-tracing-an-orchestration)). `ScriptedModel` implements the SDK's `Model` interface, so the real `Runner`, tools, agents-as-tools, handoffs, structured outputs and tracing all run. Only the model's judgment is replaced.

The helpers below wrap it in a **policy** style: `policy_model(fn)` answers every model call with `fn(call)`, where `call` is the recorded `ModelCall` (its `input`, `system_instructions`, `model_settings`...). A policy returns `say("text")`, `say_json(pydantic_obj)` for an `output_type`, or `call_tool("name", **args)`. `Meter` adds simulated latency and counts how many calls are in flight, so concurrency becomes visible without spending tokens.

In [ ]:
_ids = itertools.count(1)


def say(text: str):
    """The scripted model answers with an assistant message."""
    return assistant_message(text, item_id=f"msg_{next(_ids)}")


def say_json(obj: BaseModel):
    """The scripted model answers with structured output (what `output_type` parses)."""
    return say(obj.model_dump_json())


def call_tool(name: str, **arguments):
    """The scripted model calls a tool. Agents-as-tools and handoffs are tools too."""
    n = next(_ids)
    return function_call(name, arguments, call_id=f"call_{n}", item_id=f"fc_{n}")


def policy_model(policy: Callable, *, max_calls: int = 500, usage: Usage | None = None) -> ScriptedModel:
    """A ScriptedModel that answers each call with policy(call) (sync or async), up to max_calls times."""
    return ScriptedModel([ModelStep.respond(policy) for _ in range(max_calls)], default_usage=usage)


def _text(content: Any) -> str:
    if isinstance(content, str):
        return content
    return " ".join(p.get("text", "") for p in content or [] if isinstance(p, dict))


def user_text(call: ModelCall) -> str:
    """The latest user message the model received."""
    items = call.input if isinstance(call.input, list) else [{"role": "user", "content": call.input}]
    for item in reversed(items):
        if isinstance(item, dict) and item.get("role") == "user":
            return _text(item.get("content"))
    return ""


def tool_outputs(call: ModelCall) -> list[str]:
    """Tool results in the model input (what a manager sees after its tools ran)."""
    items = call.input if isinstance(call.input, list) else []
    return [str(i.get("output")) for i in items if isinstance(i, dict) and i.get("type") == "function_call_output"]


class Meter:
    """Simulated latency plus an in-flight counter, to make concurrency visible offline."""

    def __init__(self, seconds: float = 0.2) -> None:
        self.seconds = seconds
        self.reset()

    def reset(self) -> None:
        self.in_flight = self.peak = self.calls = 0

    def wrap(self, policy: Callable) -> Callable:
        async def timed(call: ModelCall):
            self.in_flight += 1
            self.calls += 1
            self.peak = max(self.peak, self.in_flight)
            try:
                await asyncio.sleep(self.seconds)
                return policy(call)  # a policy may return an Exception: the scripted model raises it
            finally:
                self.in_flight -= 1
        return timed


echo = Agent(name="Echo", model=policy_model(lambda call: [say(f"(scripted) you said: {user_text(call)}")]))
print((await Runner.run(echo, "hello")).final_output)

## 8.1 Workflows vs agents: choosing a decomposition shape

In a **workflow**, code fixes the steps and the model fills them in. In an **agent**, the model decides the next step. Choose the cheapest shape that meets the outcome: single call, fixed pipeline, routing, parallelization, orchestrator-workers, evaluator-optimizer, or hosted multi-agent. Start with one agent, and let evals justify every split. [Theory](README.md#81-workflows-vs-agents-choosing-a-decomposition-shape)

In [ ]:
@dataclass
class Workload:
    name: str
    one_call_enough: bool    # does one focused call meet the outcome criteria?
    steps_known: bool        # are the steps known up front?
    independent_parts: bool  # are some steps independent of each other?
    known_categories: bool   # is the open decision a choice among known categories?
    clear_criteria: bool     # clear quality criteria, and feedback improves the result?


def choose_shape(w: Workload) -> str:
    """The README 8.1 shape table as code."""
    if w.one_call_enough:
        return "single call"
    if w.clear_criteria:                     # one artifact, graded against a rubric and revised
        return "evaluator-optimizer (8.6)"
    if w.steps_known:
        return "pipeline with a parallel stage (8.2 + 8.4)" if w.independent_parts else "fixed pipeline (8.2)"
    if w.known_categories:
        return "routing (8.3)"
    return "orchestrator-workers (8.5)"


WORKLOADS = [
    Workload("Summarize one support ticket in 3 bullets", True, True, False, False, False),
    Workload("Invoice: extract -> validate -> enrich from CRM -> post to ERP", False, True, False, False, False),
    Workload("Translate a release note into 6 languages, then QA each", False, True, True, False, False),
    Workload("Send each ticket to billing / technical / account handling", False, False, False, True, False),
    Workload("Add tests to a legacy codebase nobody has mapped yet", False, False, True, False, False),
    Workload("Outage notice that must pass a 4-point comms rubric", False, True, False, False, True),
]
for w in WORKLOADS:
    print(f"{w.name:<64} -> {choose_shape(w)}")

In [ ]:
def worth_a_new_specialist(tasks_overlap: bool, long_instructions: bool, many_or_similar_tools: bool,
                           eval_gain: float | None) -> str:
    """Building agents track: split only for non-overlapping tasks with long instructions or many/similar
    tools, and only when an eval shows the split helps."""
    if tasks_overlap or not (long_instructions or many_or_similar_tools):
        return "keep one agent"
    if eval_gain is None:
        return "candidate split: measure it first"
    return "split" if eval_gain > 0 else "keep one agent (the eval shows no gain)"


print(worth_a_new_specialist(False, True, False, None))
print(worth_a_new_specialist(False, False, True, +0.07))
print(worth_a_new_specialist(True, True, True, +0.10))

> **Observe:** only one workload needs a planner model. Most real workloads are pipelines or routers, where code owns the flow and each step is testable on its own. "Hosted multi-agent" is not on the list on purpose: it is a runtime choice (8.8), not a shape you start from.

> **Try it:** add a workload "investigate why checkout latency doubled after a deploy". Which flags do you set, and does the chooser agree with the reasoning in [coverage Q1](coverage/README.md#more-self-check-questions) (steps emerge from findings)?

## 8.2 Fixed pipelines (prompt chaining) with the Agents SDK

Each step is an `Agent` with one job. Code passes one step's output to the next, and a **gate** (a structured `output_type` checked by an `if`) decides whether to continue. All steps sit inside one `trace(...)`. [Theory](README.md#82-fixed-pipelines-prompt-chaining-with-the-agents-sdk)

In [ ]:
class OutlineCheck(BaseModel):
    covers_all_sections: bool
    missing: list[str]


REQUIRED_SECTIONS = ["rollback", "data backfill", "cutover", "customer comms"]

outliner = Agent(name="Outliner", model=MODEL,
                 instructions="Outline a migration runbook for the system described. One numbered line per section.")
checker = Agent(name="Outline checker", model=MODEL, output_type=OutlineCheck,
                instructions="Check the outline has: rollback, data backfill, cutover, customer comms. List what is missing.")
writer = Agent(name="Runbook writer", model=MODEL,
               instructions="Write the runbook from the approved outline. Under 200 words.")


async def runbook_pipeline(system: str, outliner: Agent = outliner, checker: Agent = checker,
                           writer: Agent = writer) -> str:
    with trace("Runbook pipeline"):                               # all steps in ONE trace
        outline = await Runner.run(outliner, system)
        verdict = (await Runner.run(checker, outline.final_output)).final_output_as(OutlineCheck)
        if not verdict.covers_all_sections:                       # gate: code decides, not the model
            raise ValueError(f"outline rejected, missing: {verdict.missing}")
        draft = await Runner.run(writer, outline.final_output)
        return draft.final_output

In [ ]:
# Offline stand-ins. The scripted outliner "forgets" rollback when the system has no backups yet.
def outliner_policy(call: ModelCall) -> list:
    sections = ["Data backfill", "Cutover", "Customer comms"]
    if "no backups" not in user_text(call).lower():
        sections.insert(0, "Rollback")
    return [say("\n".join(f"{i}. {s}" for i, s in enumerate(sections, 1)))]


def checker_policy(call: ModelCall) -> list:
    outline = user_text(call).lower()
    missing = [s for s in REQUIRED_SECTIONS if s not in outline]
    return [say_json(OutlineCheck(covers_all_sections=not missing, missing=missing))]


offline_steps = {
    "outliner": outliner.clone(model=policy_model(outliner_policy)),
    "checker": checker.clone(model=policy_model(checker_policy)),
    "writer": writer.clone(model=policy_model(lambda call: [say("RUNBOOK\n" + user_text(call))])),
}

print(await runbook_pipeline("Orders service: Postgres 13 -> 16, 2 TB", **offline_steps))
try:
    await runbook_pipeline("Billing DB, no backups yet", **offline_steps)
except ValueError as exc:
    print("\ngate:", exc)
print("writer was called", len(offline_steps["writer"].model.calls), "time(s) for 2 pipeline runs")

In [ ]:
# (live, ~$0.002) the same pipeline on MODEL. A real checker may reject a real outline: that is the gate working.
if not RUN_LIVE_CELLS:
    print("skipped (live): set RUN_LIVE_CELLS = True in the setup section to run this cell")
else:
    try:
        print(await runbook_pipeline("Orders service: Postgres 13 -> 16, 2 TB, 99.9% SLA, Saturday maintenance window"))
    except ValueError as exc:
        print("gate rejected the live outline:", exc)

> **Observe:** the rejected run never reached the writer. The `if` runs every time; an instruction such as "stop if the outline is incomplete" would hold only most of the time. Each step can also be tested alone: `checker_policy` is exactly the behavior you would write evals for.

> **Try it:** the writer currently sees only the outline. Make it also receive the original request: pass `f"<request>{system}</request>\n<outline>{outline.final_output}</outline>"`. Nothing in the SDK carries step 1's input to step 3 for you.

## 8.3 Routing: by code, by handoff, and by model tier

Three ways to route, from most to least deterministic: a **classifier agent plus a dispatch table** in code, **handoffs** (the specialist owns the reply), and **model tiering** (cheap model first, escalate on a failed check). [Theory](README.md#83-routing-by-code-by-handoff-and-by-model-tier)

In [ ]:
class Route(BaseModel):
    category: Literal["billing", "technical", "account", "other"]
    complexity: Literal["simple", "complex"]


router = Agent(name="Router", output_type=Route, model=MODEL,
               instructions="Classify the support ticket. complex = needs investigation across systems or a senior engineer.")


def specialist(name: str, model: str) -> Agent:
    return Agent(name=name, model=model, instructions=f"You are the {name} specialist. Resolve the ticket in 3 sentences.")


SPECIALISTS = {
    ("technical", "complex"): specialist("Senior technical", STRONG_MODEL),
    ("technical", "simple"): specialist("Technical", MODEL),
    ("billing", "simple"): specialist("Billing", MODEL),
}
FALLBACK = specialist("Generalist", MODEL)


async def handle(ticket: str, router: Agent = router, specialists: dict = SPECIALISTS,
                 fallback: Agent = FALLBACK) -> tuple[str, str]:
    route = (await Runner.run(router, ticket)).final_output_as(Route)
    agent = specialists.get((route.category, route.complexity), fallback)  # unknown combos never crash
    return agent.name, (await Runner.run(agent, ticket)).final_output

In [ ]:
def router_policy(call: ModelCall) -> list:
    t = user_text(call).lower()
    category = ("billing" if any(w in t for w in ("invoice", "charged", "refund")) else
                "technical" if any(w in t for w in ("error", "500", "timeout", "webhook")) else
                "account" if any(w in t for w in ("password", "login", "seat")) else "other")
    complexity = "complex" if any(w in t for w in ("intermittent", "since the upgrade", "several regions")) else "simple"
    return [say_json(Route(category=category, complexity=complexity))]


offline_router = router.clone(model=policy_model(router_policy))
offline_specialists = {
    key: agent.clone(model=policy_model(lambda call, n=agent.name: [say(f"[{n}] on it: {user_text(call)[:40]}")]))
    for key, agent in SPECIALISTS.items()
}
offline_fallback = FALLBACK.clone(model=policy_model(lambda call: [say("[Generalist] let me look into that.")]))

for ticket in ("I was charged twice on invoice INV-7", "Webhook returns 500", "Intermittent timeouts since the upgrade",
               "Locked out after a password reset in several regions"):
    who, reply = await handle(ticket, offline_router, offline_specialists, offline_fallback)
    print(f"{ticket:<55} -> {who:<17} | {reply}")

In [ ]:
# Grade the classifier ALONE on a labeled set. No specialist runs, so this eval is cheap and fast.
LABELED = [
    ("I was charged twice on invoice INV-7", Route(category="billing", complexity="simple")),
    ("Webhook returns 500", Route(category="technical", complexity="simple")),
    ("Intermittent timeouts since the upgrade", Route(category="technical", complexity="complex")),
    ("How do I add a seat for a new hire?", Route(category="account", complexity="simple")),
    ("Can I get my money back for last month?", Route(category="billing", complexity="simple")),
]
hits = 0
for ticket, expected in LABELED:
    got = (await Runner.run(offline_router, ticket)).final_output_as(Route)
    hits += got == expected
    print(f"{'✓' if got == expected else '✗'} {ticket:<45} got {got.category}/{got.complexity}")
print(f"router accuracy: {hits}/{len(LABELED)}")

> **Observe:** the keyword stand-in misses "money back". That miss is what a labeled routing eval is for: you find it without running a single specialist. The dispatch table also never crashes on an unexpected `(account, complex)` route; it falls back.

Now the same routing decision as **handoffs**: the triage model calls `transfer_to_<agent>`, and the chosen specialist writes the reply. The routing signal is each specialist's `handoff_description`.

In [ ]:
billing_desk = Agent(name="Billing desk", handoff_description="Invoices, charges and refunds. Not technical errors.",
                     model=policy_model(lambda call: [say("Billing desk: I've opened a refund review.")]))
tech_desk = Agent(name="Technical desk", handoff_description="API errors, webhooks, outages. Not billing.",
                  model=policy_model(lambda call: [say("Technical desk: send me the request ID, please.")]))


def triage_policy(call: ModelCall) -> list:
    t = user_text(call).lower()
    return [call_tool("transfer_to_billing_desk" if "refund" in t or "charged" in t else "transfer_to_technical_desk")]


triage = Agent(name="Triage", instructions="Hand off to exactly one specialist.", handoffs=[billing_desk, tech_desk],
               model=policy_model(triage_policy))

for h in (handoff(billing_desk), handoff(tech_desk)):
    print(f"tool the model sees: {h.tool_name:<28} | {h.tool_description}")
for ticket in ("I want a refund for March", "Webhook returns 500"):
    result = await Runner.run(triage, ticket)
    print(f"\n{ticket} -> last_agent={result.last_agent.name}\n  {result.final_output}")

Finally, **model tiering**: route by difficulty instead of topic. The cheap tier answers first; a **code check** (not self-rated confidence) decides whether to escalate.

In [ ]:
KB = {"KB-101": "Rotate an API key", "KB-202": "Webhook retry policy", "KB-303": "Regional failover runbook"}


class Resolution(BaseModel):
    kb_article: str
    steps: list[str]


tier1 = Agent(name="Tier 1 (gpt-6-luna)", output_type=Resolution, model=MODEL,
              instructions=f"Resolve with one article from this KB: {KB}. Cite its ID.")
tier2 = tier1.clone(name="Tier 2 (gpt-6.1-sol)", model=STRONG_MODEL)


async def cascade(ticket: str, tiers: list[Agent]) -> tuple[str, Resolution | None]:
    for agent in tiers:
        try:
            resolution = (await Runner.run(agent, ticket)).final_output_as(Resolution)
        except ModelBehaviorError:          # invalid structured output: escalate
            continue
        if resolution.kb_article in KB:     # code check: the cited article must exist
            return agent.name, resolution
    return "human queue", None


def tier1_policy(call: ModelCall) -> list:
    t = user_text(call).lower()
    kb = "KB-101" if "api key" in t else "KB-202" if "webhook" in t else "KB-999"  # cheap tier guesses on hard ones
    return [say_json(Resolution(kb_article=kb, steps=["..."]))]


def tier2_policy(call: ModelCall) -> list:
    t = user_text(call).lower()
    kb = "KB-303" if "region" in t or "failover" in t else "KB-101" if "key" in t else "KB-999"
    return [say_json(Resolution(kb_article=kb, steps=["..."]))]


tiers = [tier1.clone(model=policy_model(tier1_policy)), tier2.clone(model=policy_model(tier2_policy))]
TICKETS = ["Leaked API key in a repo", "Webhook retries stopped", "API key expired", "Webhook signature fails",
           "Traffic stuck in one region after failover", "Our CFO wants a custom contract clause"]
handled_by = []
for ticket in TICKETS:
    who, res = await cascade(ticket, tiers)
    handled_by.append(who)
    print(f"{ticket:<45} -> {who:<22} {res.kb_article if res else ''}")

# Rough cost per ticket with 1,500 input + 300 output tokens per call (prices from the README).
luna, sol = 1500 * 0.10e-6 + 300 * 0.50e-6, 1500 * 2e-6 + 300 * 10e-6
escalated = sum(who != "Tier 1 (gpt-6-luna)" for who in handled_by)
cascade_cost = len(TICKETS) * luna + escalated * sol
print(f"\ncascade: ${cascade_cost * 100_000 / len(TICKETS):,.0f} per 100k tickets | all on sol: ${sol * 100_000:,.0f}")

In [ ]:
# (live, ~$0.001) the real router on MODEL. Compare its labels with the scripted ones above.
if not RUN_LIVE_CELLS:
    print("skipped (live): set RUN_LIVE_CELLS = True in the setup section to run this cell")
else:
    for ticket, _ in LABELED[:3]:
        route = (await Runner.run(router, ticket)).final_output_as(Route)
        print(f"{ticket:<45} -> {route.category}/{route.complexity}")

> **Observe:** handoffs moved ownership (`last_agent` is the specialist); the classifier route kept your code in charge. The cascade only paid for the strong tier on the tickets that failed the KB check, and sent the one neither tier could resolve to humans instead of looping.

> **Try it:** make `tier1_policy` return a malformed `Resolution` for one ticket (for example `say('{"kb_article": 7}')`). The `ModelBehaviorError` branch escalates it. Which is safer: escalating on invalid output or retrying the cheap tier?

## 8.4 Parallelization: fan-out, voting, and parallel tool calls

**Sectioning** splits independent work across workers; **voting** runs the same task several times and picks. Production fan-out needs a **concurrency bound** (fewer bursts against rate limits; RPM/TPM still need retries with backoff) and **per-slot failure handling**. Inside one agent, the model can also emit several tool calls in a turn, and the SDK runs them concurrently unless you cap it. [Theory](README.md#84-parallelization-fan-out-voting-and-parallel-tool-calls)

In [ ]:
async def fan_out(agent: Agent, inputs: list[str], limit: int = 4) -> list[str | Exception]:
    gate = asyncio.Semaphore(limit)                    # bounds requests in flight (bursts), not RPM / TPM

    async def one(text: str) -> str:
        async with gate:
            result = await Runner.run(agent, text, max_turns=4)
            return result.final_output

    # return_exceptions=True: a failed worker becomes an Exception in its slot, the others survive
    return await asyncio.gather(*(one(t) for t in inputs), return_exceptions=True)


meter = Meter(seconds=0.2)


def facet_policy(call: ModelCall):
    facet = user_text(call)
    if facet == "compliance":
        return TimeoutError("worker timed out")       # one worker fails
    return [say(f"{facet}: looks fine")]


facet_worker = Agent(name="Facet worker", instructions="Assess ONE facet of the vendor.",
                     model=policy_model(meter.wrap(facet_policy)))
FACETS = ["security", "data residency", "pricing", "SLA", "support", "roadmap", "operations", "compliance"]

rows = []
meter.reset()
t0 = time.perf_counter()
sequential = []
for f in FACETS:
    try:
        sequential.append((await Runner.run(facet_worker, f)).final_output)
    except Exception as exc:  # noqa: BLE001
        sequential.append(exc)
rows.append(("sequential loop", time.perf_counter() - t0, meter.peak))

meter.reset()
t0 = time.perf_counter()
unbounded = await asyncio.gather(*(Runner.run(facet_worker, f) for f in FACETS), return_exceptions=True)
rows.append(("gather, unbounded", time.perf_counter() - t0, meter.peak))

meter.reset()
t0 = time.perf_counter()
bounded = await fan_out(facet_worker, FACETS, limit=3)
rows.append(("fan_out(limit=3)", time.perf_counter() - t0, meter.peak))

for name, seconds, peak in rows:
    print(f"{name:<20} {seconds:5.2f}s   peak in flight: {peak}")

In [ ]:
def summarize_slots(names: list[str], results: list, blocking: bool) -> str:
    """Per-slot policy: a research brief reports gaps; a security review blocks on any gap."""
    gaps = {n: type(r).__name__ for n, r in zip(names, results) if isinstance(r, Exception)}
    covered = len(names) - len(gaps)
    if gaps and blocking:
        return f"BLOCKED: {covered}/{len(names)} reviewed, unreviewed: {gaps}"
    return f"{covered}/{len(names)} facets covered" + (f"; gaps reported: {gaps}" if gaps else "")


print("research brief :", summarize_slots(FACETS, bounded, blocking=False))
print("security review:", summarize_slots(FACETS, bounded, blocking=True))

try:
    await asyncio.gather(*(Runner.run(facet_worker, f) for f in FACETS))  # no return_exceptions
except TimeoutError as exc:
    print(f"\nbare gather raised {exc!r}: the 7 good results are gone")

**Voting (best-of-n).** The SDK's `parallelization.py` example runs a translator three times in parallel and lets a picker choose. The picker is a separate agent with its own criteria.

In [ ]:
class Pick(BaseModel):
    rationale: str
    best: int


variants = itertools.cycle(["Clé d'API révoquée.", "Votre API key a été révoquée.", "La clé API a été révoquée."])
translator = Agent(name="Translator", instructions="Translate to French. Keep product terms.",
                   model=policy_model(lambda call: [say(next(variants))]))


def picker_policy(call: ModelCall) -> list:
    options = re.findall(r"^(\d+)\. (.*)$", user_text(call), re.M)
    best = next(int(i) for i, text in options if text.startswith("La clé API"))
    return [say_json(Pick(rationale=f"Option {best} keeps the product term and reads naturally.", best=best))]


picker = Agent(name="Picker", output_type=Pick, instructions="Pick the most accurate, natural translation.",
               model=policy_model(picker_policy))

candidates = await asyncio.gather(*(Runner.run(translator, "Your API key was revoked.") for _ in range(3)))
options = "\n".join(f"{i}. {r.final_output}" for i, r in enumerate(candidates))
pick = (await Runner.run(picker, f"<options>\n{options}\n</options>")).final_output_as(Pick)
print(options, "\n->", candidates[pick.best].final_output, "|", pick.rationale)

**Parallel tool calls inside one agent.** A manager emits three `as_tool()` calls in **one** turn. `ModelSettings(parallel_tool_calls=True)` is what the model is allowed to do; `ToolExecutionConfig(max_function_tool_concurrency=N)` is how many of those calls the SDK runs at once.

In [ ]:
tool_meter = Meter(seconds=0.3)
area_reviewer = Agent(name="Area reviewer", instructions="Review the change for ONE area.",
                      model=policy_model(tool_meter.wrap(lambda call: [say(f"{user_text(call)}: no blocking issues")])))


def manager_policy(call: ModelCall) -> list:
    outputs = tool_outputs(call)
    if not outputs:  # first turn: three tool calls in ONE model response
        return [call_tool("review_area", input=area) for area in ("security", "performance", "tests")]
    return [say("Merged review:\n- " + "\n- ".join(outputs))]


review_manager = Agent(
    name="Review manager",
    model=policy_model(manager_policy),
    model_settings=ModelSettings(parallel_tool_calls=True),
    tools=[area_reviewer.as_tool(tool_name="review_area", tool_description="Review one area of the change.")],
)

for cap in (None, 1, 3):
    tool_meter.reset()
    t0 = time.perf_counter()
    result = await Runner.run(review_manager, "Review PR #42",
                              run_config=RunConfig(tool_execution=ToolExecutionConfig(max_function_tool_concurrency=cap)))
    print(f"max_function_tool_concurrency={cap!s:<5} {time.perf_counter() - t0:4.2f}s  peak nested runs: {tool_meter.peak}")
print("\nparallel_tool_calls sent to the model:", review_manager.model.first_call.model_settings.parallel_tool_calls)
print(result.final_output)

> **Observe:** wall time tracks the slowest batch, not the sum. Unbounded `gather` hit 8 in flight, which against a real per-minute rate limit is how a nightly job turns into a burst of 429s. Inside the manager, the SDK started all three nested agents at once by default (`None`); the cap of 1 serialized them.

> **Try it:** set the `Meter` to 1.0 s and run `fan_out` with `limit=8` and `limit=2`. Then put the whole fan-out behind a check: if nobody needs the result interactively, would the Batch API from [Chapter 07](../07-batch-and-cost/README.md) be cheaper?

## 8.5 Dynamic decomposition: orchestrator-workers

When the subtasks depend on the input, a planner decides them. **Option A:** the planner returns a structured `Plan`, code validates its **coverage**, runs the workers in parallel and synthesizes. **Option B:** a manager agent plans and dispatches through `Agent.as_tool()` workers. Either way, each worker sees **only** the context you pass it. [Theory](README.md#85-dynamic-decomposition-orchestrator-workers)

In [ ]:
class Subtask(BaseModel):
    id: str
    facet: str = Field(description="Which required facet this subtask covers")
    question: str
    context: str = Field(description="Every fact the worker needs. Workers see nothing else.")


class Plan(BaseModel):
    subtasks: list[Subtask] = Field(max_length=6)          # becomes maxItems in the strict schema


REQUIRED_FACETS = {"security", "data", "cost", "operations"}

planner = Agent(name="Planner", output_type=Plan, model=STRONG_MODEL,
                instructions="Split the request into independent subtasks, at least one per facet: "
                             "security, data, cost, operations. Put every fact a worker needs into its context.")
worker = Agent(name="Worker", model=MODEL, instructions="Answer ONE subtask using only its context. Quote the context you rely on.")
synthesizer = Agent(name="Synthesizer", model=MODEL, instructions="Merge findings by facet. Report gaps; never invent.")


async def orchestrate(request: str, planner: Agent = planner, worker: Agent = worker,
                      synthesizer: Agent = synthesizer) -> str:
    plan = (await Runner.run(planner, request)).final_output_as(Plan)
    missing = REQUIRED_FACETS - {s.facet for s in plan.subtasks}
    if missing:                                                    # catches over-narrow decomposition
        raise ValueError(f"plan too narrow, missing facets: {sorted(missing)}")
    prompts = [f"<subtask id='{s.id}'>\n{s.question}\n<context>{s.context}</context>\n</subtask>" for s in plan.subtasks]
    results = await fan_out(worker, prompts)
    findings = "\n".join(
        f"<finding id='{s.id}' facet='{s.facet}'>{r}</finding>" if not isinstance(r, Exception)
        else f"<finding id='{s.id}' facet='{s.facet}' status='failed'>{type(r).__name__}</finding>"
        for s, r in zip(plan.subtasks, results)
    )
    return (await Runner.run(synthesizer, findings)).final_output


print("strict schema for Plan.subtasks:", AgentOutputSchema(Plan).json_schema()["properties"]["subtasks"])

In [ ]:
VENDOR_FACTS = {
    "security": "SOC 2 Type II report dated 2026-03. SAML SSO. No penetration-test summary shared.",
    "data": "Customer data stored in eu-west-1. Three subprocessors, all EU.",
    "cost": "$48k/year list price, 3-year term, 7% annual uplift.",
    "operations": "99.9% SLA, 4-hour P1 response, public status page.",
}


def plan_of(*facets: str) -> Plan:
    return Plan(subtasks=[Subtask(id=f"s{i}", facet=f, question=f"Assess {f} for vendor Acme.",
                                  context=VENDOR_FACTS[f]) for i, f in enumerate(facets, 1)])


def worker_policy(call: ModelCall):
    text = user_text(call)
    if "99.9% SLA" in text:
        return TimeoutError("operations worker timed out")
    quote = re.search(r"<context>(.*?)</context>", text, re.S).group(1)
    return [say(f'Based on "{quote[:40]}...": acceptable with follow-ups.')]


def synth_policy(call: ModelCall) -> list:
    text = user_text(call)
    failed = re.findall(r"facet='(\w+)' status='failed'", text)
    done = len(re.findall(r"<finding ", text)) - len(failed)
    return [say(f"Merged {done} facet findings. Gaps: {failed or 'none'}.")]


narrow_planner = planner.clone(model=policy_model(lambda call: [say_json(plan_of("security", "data"))]))
good_planner = planner.clone(model=policy_model(lambda call: [say_json(plan_of("security", "data", "cost", "operations"))]))
offline_worker = worker.clone(model=policy_model(worker_policy))
offline_synth = synthesizer.clone(model=policy_model(synth_policy))

try:
    await orchestrate("Assess vendor Acme", narrow_planner, offline_worker, offline_synth)
except ValueError as exc:
    print("coverage check:", exc)

print("synthesis:", await orchestrate("Assess vendor Acme", good_planner, offline_worker, offline_synth))
first_worker_input = user_text(offline_worker.model.first_call)
print("\nwhat one worker saw:\n" + first_worker_input)
print("\nleaked other facets into it?", any(v in first_worker_input for k, v in VENDOR_FACTS.items() if k != "security"))

In [ ]:
# The schema caps the plan size. A planner that returns 7 subtasks fails validation instead of fanning out 7 workers.
too_big = Plan.model_construct(subtasks=[Subtask(id=str(i), facet="security", question="q", context="c") for i in range(7)])
greedy_planner = planner.clone(model=policy_model(lambda call: [say(too_big.model_dump_json())]))
try:
    await Runner.run(greedy_planner, "Assess vendor Acme")
except ModelBehaviorError as exc:
    print("ModelBehaviorError:", exc)   # the SDK reports any schema failure (here: 7 > maxItems 6) as "Invalid JSON"

**Option B: a manager with workers as tools.** `parameters=ReviewTask` gives the tool a structured input schema, so code and reviewers can see exactly what the manager passed. The nested agent receives that input as JSON, and nothing from the manager's conversation.

In [ ]:
class ReviewTask(BaseModel):
    path: str
    diff: str
    focus: Literal["security", "correctness", "tests"]


file_reviewer = Agent(name="File reviewer", instructions="Review one file's diff for the given focus only. Cite line numbers.",
                      model=policy_model(lambda call: [say(f"reviewed: {user_text(call)[:60]}...")]))


def review_manager_policy(call: ModelCall) -> list:
    if not tool_outputs(call):
        return [call_tool("review_file", path="app/auth.py", diff="+ sql = f'... {name}'", focus="security"),
                call_tool("review_file", path="app/db.py", diff="+ def fetch_user(user_id)", focus="correctness")]
    return [say(f"One merged review from {len(tool_outputs(call))} file reviews.")]


manager = Agent(
    name="Review manager", model=policy_model(review_manager_policy),
    instructions="Call review_file once per changed file and focus. Independent calls can go in the same turn. "
                 "Then write one merged review; drop duplicates.",
    tools=[file_reviewer.as_tool(tool_name="review_file", tool_description="Review one file diff for one focus area.",
                                 parameters=ReviewTask)],
)
print("tool schema properties:", list(manager.tools[0].params_json_schema["properties"]))
result = await Runner.run(manager, "Here is my whole conversation context: customer=ACME, ticket=SEC-9, PR #42 ...")
print("final:", result.final_output, "| last_agent:", result.last_agent.name)
print("nested agent input:", file_reviewer.model.first_call.input)

**Adaptive re-planning with a cap.** The "add tests to a legacy codebase" task from [coverage Q1](coverage/README.md#more-self-check-questions): the mapping step finds a shared fixture database with no seed data, so the next plan adds a seed step. The planner sees the results so far and emits the *next* plan; an empty plan means done; a round counter bounds it. Subtasks inside one round run in parallel, so they must be independent: the tests depend on the seed data, so they go in the **next** round, with the seed step's result passed in their `context`.

In [ ]:
async def adaptive(request: str, planner: Agent, worker: Agent, max_rounds: int = 4) -> tuple[list[str], int, bool]:
    findings: list[str] = []
    for round_no in range(1, max_rounds + 1):
        so_far = "\n".join(findings) or "(none yet)"
        plan = (await Runner.run(planner, f"<request>{request}</request>\n<findings>\n{so_far}\n</findings>")).final_output_as(Plan)
        if not plan.subtasks:                       # the planner says the work is complete
            return findings, round_no, True
        results = await fan_out(worker, [f"{s.question}\n<context>{s.context}</context>" for s in plan.subtasks])
        findings += [f"[{s.id}] {r}" for s, r in zip(plan.subtasks, results)]
    return findings, max_rounds, False              # cap reached: report what is unfinished


def legacy_planner_policy(call: ModelCall) -> list:
    seen = user_text(call)
    step = lambda i, q, ctx="repo: billing-legacy": Subtask(id=i, facet="tests", question=q, context=ctx)  # noqa: E731
    if "(none yet)" in seen:
        return [say_json(Plan(subtasks=[step("map", "Map modules and external dependencies.")]))]
    if "fixture database" in seen and "[seed]" not in seen:      # the tests need seed data: plan the seed alone
        return [say_json(Plan(subtasks=[step("seed", "Write a seed script for the fixture database.")]))]
    if "[seed]" in seen and "[tests]" not in seen:               # next round: hand the seed result to the tests
        seed_result = re.search(r"\[seed\] (.*)", seen).group(1)
        return [say_json(Plan(subtasks=[step("tests", "Write tests for invoice.py on the seeded database.",
                                             f"repo: billing-legacy; {seed_result}")]))]
    return [say_json(Plan(subtasks=[]))]


def legacy_worker_policy(call: ModelCall) -> list:
    q = user_text(call)
    if q.startswith("Map"):
        return [say("found a shared fixture database with no seed data: billing_fixtures")]
    if q.startswith("Write a seed"):
        return [say("seed ready: tests/fixtures/seed_billing.sql")]
    if "seed ready" not in q:                                    # a dependent step without its input cannot succeed
        return [say("blocked: no seed data in my context")]
    return [say(f"done: {q.splitlines()[0]}")]


legacy_planner = planner.clone(model=policy_model(legacy_planner_policy))
legacy_worker = worker.clone(model=policy_model(legacy_worker_policy))
for cap in (4, 1):
    findings, rounds, complete = await adaptive("Add tests to billing-legacy", legacy_planner, legacy_worker, max_rounds=cap)
    print(f"max_rounds={cap}: complete={complete} after {rounds} round(s)")
    print("  " + "\n  ".join(findings))
tests_input = next(user_text(c) for c in legacy_worker.model.calls if user_text(c).startswith("Write tests"))
assert "tests/fixtures/seed_billing.sql" in tests_input, tests_input
print("\nthe tests worker received the seed script in its context:", tests_input.splitlines()[-1])

> **Observe:** the narrow plan failed at the plan seam, before any worker ran. That is the over-narrow decomposition trap caught by code, not by a stronger worker. The `as_tool` worker never saw "customer=ACME"; it only saw the structured `ReviewTask`. The adaptive loop ran map → seed → tests in three rounds and needed a fourth to hear "done"; the tests ran only after the seed data existed, and got the seed script in their context. With `max_rounds=1` the loop stopped and said so, instead of running forever.

> **Try it:** build `plan = plan_of("security", "data", "cost", "operations")`, set `plan.subtasks[0].facet = "Security"`, and make `narrow_planner` return it (`say_json(plan)`). The set check now fails on a facet that is present. Normalize facets (or make `facet` a `Literal`) so the coverage check tests meaning, not spelling.

**Shared memory keyed by a conversation id.** When agents take turns on the *same* conversation (triage, then research, then drafting), the Building agents track says to share memory keyed by `conversation_id` instead of copying transcripts by hand. In the SDK that is a `Session`. Below, a registry hands out one `SQLiteSession` per conversation id (in-memory here; pass a file path to persist it). The drafter sees the researcher's turn without any glue code. A different conversation id sees nothing, and the independent reviewer gets no session at all. [Theory](README.md#85-dynamic-decomposition-orchestrator-workers)

In [ ]:
from agents import SQLiteSession

_MEMORY: dict[str, SQLiteSession] = {}


def memory_for(conversation_id: str) -> SQLiteSession:
    """One shared Session per conversation id. Every agent that continues this conversation uses it."""
    return _MEMORY.setdefault(conversation_id, SQLiteSession(conversation_id))   # ':memory:' by default


researcher = Agent(name="Researcher", model=policy_model(lambda call: [say("finding: vendor X stores data in eu-west-1")]))
drafter_mem = Agent(name="Drafter", model=policy_model(
    lambda call: [say(f"draft built on {len(call.input) if isinstance(call.input, list) else 1} input items")]))
reviewer_mem = Agent(name="Independent reviewer", model=policy_model(
    lambda call: [say(f"reviewed with {len(call.input) if isinstance(call.input, list) else 1} input item(s)")]))

conv = "conv_vendor_x_42"
await Runner.run(researcher, "Where does vendor X store customer data?", session=memory_for(conv))
drafted = await Runner.run(drafter_mem, "Draft the data-residency paragraph.", session=memory_for(conv))
print("drafter:", drafted.final_output)
seen_roles = [i.get("role") for i in drafter_mem.model.first_call.input if isinstance(i, dict)]
print("drafter input roles:", seen_roles)
assert seen_roles == ["user", "assistant", "user"], seen_roles   # the researcher's turn arrived via the session

print("items stored for", conv, "->", len(await memory_for(conv).get_items()))
print("items stored for conv_other ->", len(await memory_for("conv_other").get_items()))

# The independent reviewer must NOT share the memory: explicit, scoped input only (README 8.9).
reviewed = await Runner.run(reviewer_mem, f"<draft>{drafted.final_output}</draft>")
print("reviewer:", reviewed.final_output)

> **Observe:** the drafter's first model call carried three items (the research question, the researcher's answer, its own request) because both runs used the session for `conv_vendor_x_42`. The other conversation id is empty. The reviewer saw one item. Shared memory is opt-in per agent: use it for agents that continue one conversation, and keep parallel workers and reviewers on explicit inputs.

## 8.6 Evaluator-optimizer loops

A generator drafts, an evaluator with an explicit rubric returns a **structured verdict**, and the generator revises with the exact feedback until the draft passes or a **cap** is reached. The evaluator gets a **fresh context**: the brief and the draft only. [Theory](README.md#86-evaluator-optimizer-loops)

In [ ]:
class Review(BaseModel):
    rationale: str                                   # generated first: reason before the verdict
    verdict: Literal["pass", "revise"]
    feedback: list[str]                              # specific, actionable items


drafter = Agent(name="Drafter", model=MODEL, instructions="Write the customer notice. Apply any feedback you are given.")
evaluator = Agent(name="Evaluator", model=STRONG_MODEL, output_type=Review,
                  instructions="Grade the notice: (1) states the outage window in UTC, (2) names affected regions, "
                               "(3) no blame on third parties, (4) under 120 words. 'pass' only if all four hold.")


async def draft_with_review(brief: str, drafter: Agent = drafter, evaluator: Agent = evaluator,
                            max_rounds: int = 3) -> tuple[str, Review, int]:
    prompt = brief
    for round_no in range(1, max_rounds + 1):        # the cap is part of the design, not an afterthought
        draft = (await Runner.run(drafter, prompt)).final_output
        review = (await Runner.run(evaluator, f"<brief>{brief}</brief>\n<draft>{draft}</draft>")).final_output_as(Review)
        if review.verdict == "pass":
            return draft, review, round_no
        prompt = (f"<brief>{brief}</brief>\n<previous_draft>{draft}</previous_draft>\n<feedback>\n"
                  + "\n".join(f"- {item}" for item in review.feedback) + "\n</feedback>")
    return draft, review, max_rounds                 # still failing: the caller routes it to a human

In [ ]:
def rubric_feedback(draft: str) -> list[str]:
    """What the scripted evaluator checks. In live mode the evaluator model applies the same rubric."""
    feedback = []
    if "UTC" not in draft:
        feedback.append("State the outage window in UTC, e.g. 09:00-09:40 UTC.")
    if not re.search(r"\b(eu|us|ap)-[a-z]+-\d\b", draft):
        feedback.append("Name the affected regions.")
    if re.search(r"(?i)provider's fault|because of our (cloud )?provider", draft):
        feedback.append("Remove the blame on the cloud provider.")
    if len(draft.split()) >= 120:
        feedback.append("Cut the notice to under 120 words.")
    return feedback


def evaluator_policy(call: ModelCall) -> list:
    draft = re.search(r"<draft>(.*)</draft>", user_text(call), re.S).group(1)
    feedback = rubric_feedback(draft)
    return [say_json(Review(rationale=f"{4 - len(feedback)}/4 criteria met.",
                            verdict="revise" if feedback else "pass", feedback=feedback))]


FIRST_DRAFT = "Yesterday 09:00-09:40 checkout was down in eu-west-1 because of our cloud provider. Sorry!"
FIXED_DRAFT = "On 2 Oct, 09:00-09:40 UTC, checkout was unavailable in eu-west-1. Orders placed then were not charged. We are sorry."


def drafter_policy(call: ModelCall) -> list:
    return [say(FIXED_DRAFT if "<feedback>" in user_text(call) else FIRST_DRAFT)]


offline_drafter = drafter.clone(model=policy_model(drafter_policy))
offline_evaluator = evaluator.clone(model=policy_model(evaluator_policy))
draft, review, rounds = await draft_with_review("Notify customers about yesterday's checkout outage.",
                                                offline_drafter, offline_evaluator)
print(f"{review.verdict} after {rounds} round(s): {draft}")
print("revision prompt the drafter got:\n" + user_text(offline_drafter.model.last_call))
evaluator_inputs = [user_text(c) for c in offline_evaluator.model.calls]
print("\nevaluator ever saw feedback or drafter prompts?",
      any("<feedback>" in x or "<previous_draft>" in x for x in evaluator_inputs))

In [ ]:
# A drafter that ignores feedback: the cap stops the loop and the caller decides what happens next.
stubborn = drafter.clone(model=policy_model(lambda call: [say(FIRST_DRAFT)]))
draft, review, rounds = await draft_with_review("Notify customers about the outage.", stubborn, offline_evaluator, max_rounds=3)
action = "send" if review.verdict == "pass" else "route to a human editor"
print(f"verdict={review.verdict} after {rounds} rounds -> {action}; open feedback: {review.feedback}")

> **Observe:** the revision prompt carries all three ingredients (brief, previous draft, exact feedback), and the evaluator never saw any of the drafter's prompts. With the stubborn drafter the loop ended at the cap with a defined outcome instead of running 14 rounds.

> **Try it:** measure it. Run `draft_with_review` over five briefs and count rounds. If most pass on round 1, the loop is cost without benefit; if every first draft fails the same criterion, fix the drafter's instructions instead of relying on the loop.

## 8.7 Orchestrating via LLM vs via code

Code-orchestration (sequence, `if`, loops, `gather`) is more predictable; LLM-orchestration (agents-as-tools, handoffs) adapts better. A good default is **code outside, LLM inside**. For decomposition, the key facts are: handoffs cannot fan out (one honored per turn) and cannot merge; agents-as-tools can do both. [Theory](README.md#87-orchestrating-via-llm-vs-via-code)

In [ ]:
def specialist_model(name: str) -> ScriptedModel:
    return policy_model(lambda call: [say(f"{name}: OK ({len(call.input)} input item(s) seen)")])


pricing, legal, security = (Agent(name=n, handoff_description=f"{n} questions", model=specialist_model(n))
                            for n in ("pricing", "legal", "security"))

# Experiment 1: the coordinator tries to "fan out" with three handoffs in one turn.
coordinator = Agent(name="Coordinator", handoffs=[pricing, legal, security], model=policy_model(
    lambda call: [call_tool("transfer_to_pricing"), call_tool("transfer_to_legal"), call_tool("transfer_to_security")]))
history = [{"role": "user", "content": "Customer: Globex, 400 seats, EU data only."},
           {"role": "assistant", "content": "Noted."},
           {"role": "user", "content": "Check pricing, legal and security for this deal and give me one memo."}]
result = await Runner.run(coordinator, history)
print("last_agent:", result.last_agent.name, "| final:", result.final_output)
for item in result.new_items:
    print("  ", item.type, getattr(item, "output", "") if item.type == "tool_call_output_item" else "")
print("calls made -> pricing:", len(pricing.model.calls), "legal:", len(legal.model.calls), "security:", len(security.model.calls))

In [ ]:
# Experiment 2: the same specialists as tools. One turn, three calls, one merged answer.
def deal_manager_policy(call: ModelCall) -> list:
    if not tool_outputs(call):
        return [call_tool(f"ask_{n}", input=f"Globex, 400 seats, EU data only: {n} view?") for n in ("pricing", "legal", "security")]
    return [say("Deal memo:\n- " + "\n- ".join(tool_outputs(call)))]


specialists = {n: Agent(name=n, model=specialist_model(n)) for n in ("pricing", "legal", "security")}
deal_manager = Agent(name="Deal manager", model=policy_model(deal_manager_policy), tools=[
    a.as_tool(tool_name=f"ask_{n}", tool_description=f"Ask the {n} specialist.") for n, a in specialists.items()])
result = await Runner.run(deal_manager, history)
print("last_agent:", result.last_agent.name)
print(result.final_output)
print("\nwhat the handoff target saw:", len(pricing.model.first_call.input), "items (the whole conversation + the handoff)")
print("what an as_tool specialist saw:", specialists["pricing"].model.first_call.input)

> **Observe:** three handoff calls, one honored: the SDK answered the extra ones with "Multiple handoffs detected, ignoring this one.", `legal` and `security` never ran, and nobody merged. With agents-as-tools all three ran in the same turn and the manager wrote one memo. Context also differs: the handoff target got the full history; the tool got one input string.

> **Try it:** give the handoff a filter, `handoff(pricing, input_filter=...)` (see `agents.extensions.handoff_filters`), and check how many items the pricing agent sees now. When would you want that?

## 8.8 Hosted subagents: Responses API Multi-agent, Agents API, Codex

OpenAI can also run the orchestration on its side. **Responses API Multi-agent** (beta) lets a root agent spawn subagents with hosted actions; **Agents API** sessions (beta) can enable subagents in the managed Codex harness; **Codex** has its own subagents. Nothing here is sent: we build the requests, check them against the installed SDK, and parse a mock multi-agent output. [Theory](README.md#88-hosted-subagents-responses-api-multi-agent-agents-api-codex)

In [ ]:
from openai import OpenAI
from openai.types.beta.response_create_params import MultiAgent

get_file_tool = {
    "type": "function", "name": "get_file", "strict": True,
    "description": "Return the contents of one file in the pull request.",
    "parameters": {"type": "object", "properties": {"path": {"type": "string"}},
                   "required": ["path"], "additionalProperties": False},
}
multi_agent_request = {
    "model": STRONG_MODEL,                            # documented for gpt-6.1-sol and GPT-5.6 models
    "input": "Review PR #42 with three agents: correctness, security, missing tests. Reconcile duplicates.",
    "tools": [get_file_tool],                         # EVERY agent in the tree may call it
    "multi_agent": {"enabled": True, "max_concurrent_subagents": 3},   # counts the whole tree except /root
    "betas": ["responses_multi_agent=v1"],            # beta header; item schemas may change
    "store": False,
}

offline_client = OpenAI(api_key="sk-offline-inspection-only")    # never used to send anything
accepted = set(inspect.signature(offline_client.beta.responses.create).parameters)
print("unknown top-level fields:", set(multi_agent_request) - accepted or "none")
print("unknown multi_agent fields:", set(multi_agent_request["multi_agent"]) - set(MultiAgent.__annotations__) or "none")
print("betas annotation:", inspect.signature(offline_client.beta.responses.create).parameters["betas"].annotation)


def multi_agent_warnings(request: dict) -> list[str]:
    """Limits from the Responses Multi-agent guide, checked before you spend a request."""
    warnings = []
    if not (request["model"] == "gpt-6.1-sol" or request["model"].startswith("gpt-5.6")):
        warnings.append(f"{request['model']}: Multi-agent is documented for gpt-6.1-sol and GPT-5.6 models")
    if (request.get("reasoning") or {}).get("summary"):
        warnings.append("reasoning.summary is not supported with Multi-agent")
    if "max_tool_calls" in request:
        warnings.append("max_tool_calls is not supported with Multi-agent")
    return warnings


print("\nour request:", multi_agent_warnings(multi_agent_request) or "no warnings")
print("a careless one:", multi_agent_warnings({**multi_agent_request, "model": MODEL, "max_tool_calls": 5,
                                               "reasoning": {"summary": "auto"}}))

In [ ]:
from pydantic import TypeAdapter
from openai.types.beta import BetaResponseOutputItem

# The shape of a multi-agent response (based on the guide's examples), parsed into the SDK's beta types.
MOCK_OUTPUT = [
    {"type": "multi_agent_call", "id": "mac_1", "call_id": "call_spawn_sec", "action": "spawn_agent",
     "arguments": json.dumps({"task_name": "security", "fork_turns": "all", "message": "enc_..."}),
     "agent": {"agent_name": "/root"}},
    {"type": "multi_agent_call_output", "id": "maco_1", "call_id": "call_spawn_sec", "action": "spawn_agent",
     "output": [{"type": "output_text", "text": '{"task_name":"/root/security"}', "annotations": [], "logprobs": []}],
     "agent": {"agent_name": "/root"}},
    {"type": "function_call", "id": "fc_1", "call_id": "call_get_auth", "name": "get_file",
     "arguments": '{"path": "app/auth.py"}', "status": "completed", "agent": {"agent_name": "/root/security"}},
    {"type": "agent_message", "id": "amsg_1", "author": "/root/security", "recipient": "/root",
     "content": [{"type": "encrypted_content", "encrypted_content": "enc_..."}], "agent": {"agent_name": "/root"}},
    {"type": "message", "id": "msg_1", "role": "assistant", "status": "completed", "phase": "final_answer",
     "agent": {"agent_name": "/root"}, "content": [{"type": "output_text", "text": "P0: SQL injection in app/auth.py:11", "annotations": []}]},
]
items = [TypeAdapter(BetaResponseOutputItem).validate_python(raw) for raw in MOCK_OUTPUT]


def agent_name(item) -> str:
    return item.agent.agent_name if getattr(item, "agent", None) else "/root"


def pending_function_calls(items: list) -> list:
    """Your tools, whichever agent in the tree called them. multi_agent_call items are NOT yours to run."""
    return [i for i in items if i.type == "function_call"]


def final_answer(items: list) -> str:
    return "".join(p.text for i in items if i.type == "message" and agent_name(i) == "/root"
                   and i.phase == "final_answer" for p in i.content if p.type == "output_text")


for i in items:
    # For an agent_message, item.agent is the RECIPIENT; author -> recipient shows the direction.
    who = f"{i.author} -> {i.recipient}" if i.type == "agent_message" else f"by {agent_name(i)}"
    print(f"{i.type:<24} {who:<26} -> {'execute it' if i.type == 'function_call' else 'hosted / read only'}")

# Continuation over HTTP with store=False: nothing is kept server-side, so the next request must carry the
# ORIGINAL input, every output item, and your function outputs (as in the guide's HTTP example).
history = [{"role": "user", "content": multi_agent_request["input"]}]
history.extend(items)
for call in pending_function_calls(items):
    history.append({"type": "function_call_output", "call_id": call.call_id,
                    "output": json.dumps({"path": json.loads(call.arguments)["path"], "content": "..."})})
print("\ncontinuation input starts with:", history[0])
print("continuation input ends with:", history[-1])
print("final answer:", final_answer(items))

In [ ]:
# Agents API subagents (managed Codex harness): no function tools in subagents; they inherit MCP tools and web search.
from openai.types.beta.multi_agent_config_param import MultiAgentConfigParam

session_request = {
    "agent": {
        "model": STRONG_MODEL,
        "instructions": "Review the repository's open PR. Delegate independent areas to subagents.",
        "tools": [{"type": "web_search"}],
        "multi_agent": {"enabled": True, "max_concurrent_subagents": 4},   # default 6, coordinator excluded
    },
    "environment": {"type": "none"},
    "input": "Review PR #42.",
}
accepted = set(inspect.signature(offline_client.beta.agents.sessions.create).parameters)
print("unknown session fields:", set(session_request) - accepted or "none")
print("unknown multi_agent fields:", set(session_request["agent"]["multi_agent"]) - set(MultiAgentConfigParam.__annotations__) or "none")

# The Agents SDK bridge to Responses Multi-agent is documented as EXPERIMENTAL (SDK Models page). It uses the
# Responses WebSocket transport, rejects reasoning.summary / max_tool_calls / your own multi_agent or betas
# overrides, and one instance owns at most one active hosted response. Inspect it; don't build on it yet.
from agents.extensions.experimental.hosted_multi_agent import OpenAIHostedMultiAgentModel

print("\nOpenAIHostedMultiAgentModel", inspect.signature(OpenAIHostedMultiAgentModel.__init__))

> **Observe:** of five output items, exactly one is yours to execute: the `function_call` from `/root/security`. The hosted `multi_agent_call` already has its `multi_agent_call_output`, and `agent_message` content is encrypted. The validator also caught the careless request (wrong model, unsupported `max_tool_calls` and `reasoning.summary`) without a network call.

> **Try it:** the review team now wants the security subagent to run on a cheaper model and without `get_file`. Can either hosted option do that? (README 8.8 gotcha.) Sketch the same three reviewers with the Agents SDK instead.

> [!NOTE]
> Do not start new orchestrations in **Agent Builder** (shuts down 2026-11-30) or on the **Assistants API** (retired 2026-08-26). The Agents SDK is the migration target.

## 8.9 Multi-pass code review and independent reviewers

One pass over a big PR gives uneven depth. Three passes fix it: **per-file** reviews in parallel, one **integration** pass for cross-file issues, and an **independent reviewer** in a fresh context (ideally on a different model) that confirms or rejects each finding. Codex ships the same pattern as `/review`, `codex review` and `@codex review`. [Theory](README.md#89-multi-pass-code-review-and-independent-reviewers)

In [ ]:
PR_FILES: dict[str, dict[int, str]] = {   # path -> {line number: added line}
    "app/auth.py": {
        10: "def login(name: str, password: str) -> bool:",
        11: "    sql = f\"SELECT id, pw_hash FROM users WHERE name = '{name}'\"",
        12: "    row = db.execute(sql).fetchone()",
        13: "    return row is not None and check_hash(password, row.pw_hash)",
    },
    "app/db.py": {
        40: "def fetch_user(user_id: int) -> dict:          # renamed from get_user",
        41: "    return execute('SELECT * FROM users WHERE id = ?', (user_id,)).fetchone()",
    },
    "app/routes.py": {
        22: "@app.get('/users/<user_id>')",
        23: "def show_user(user_id):",
        24: "    return db.get_user(user_id)",
    },
    "tests/test_routes.py": {
        5: "def test_show_user(client):",
        6: "    assert client.get('/users/1').status_code == 200",
    },
}


def render_file(path: str) -> str:
    return f"+++ {path}\n" + "\n".join(f"{n:>4} + {code}" for n, code in PR_FILES[path].items())


DIFF = "\n".join(render_file(p) for p in PR_FILES)


class Finding(BaseModel):
    id: str
    path: str
    line: int
    severity: Literal["P0", "P1", "P2", "P3"]
    rule: str
    quote: str
    explanation: str


class FileFindings(BaseModel):
    findings: list[Finding]


class Adjudication(BaseModel):
    finding_id: str
    rationale: str                                      # reason before the decision
    decision: Literal["confirmed", "rejected", "needs_human"]


class ReviewerReport(BaseModel):
    adjudications: list[Adjudication]
    missed_issues: list[str]


print(DIFF)

In [ ]:
def F(fid, path, line, sev, rule, explanation) -> Finding:
    return Finding(id=fid, path=path, line=line, severity=sev, rule=rule, quote=PR_FILES[path].get(line, "").strip(),
                   explanation=explanation)


# Scripted per-file reviewers: what a decent model would report for each file, including one false positive.
LOCAL_FINDINGS = {
    "app/auth.py": [F("A1", "app/auth.py", 11, "P0", "sql-injection", "User input formatted into SQL."),
                    F("A2", "app/auth.py", 13, "P1", "plaintext-password", "Password compared without hashing.")],
    "app/db.py": [],
    "app/routes.py": [F("R1", "app/routes.py", 24, "P1", "missing-authz", "Any caller can read any user.")],
    "tests/test_routes.py": [F("T1", "tests/test_routes.py", 6, "P3", "weak-test", "No 404 or auth test.")],
}


def file_worker_policy(call: ModelCall) -> list:
    path = re.search(r"\+\+\+ (\S+)", user_text(call)).group(1)
    return [say_json(FileFindings(findings=LOCAL_FINDINGS[path]))]


def interfaces() -> str:
    """Public interfaces only (def lines and calls into db), for the integration pass."""
    return "\n".join(f"{p}:{n}: {c.strip()}" for p, lines in PR_FILES.items() for n, c in lines.items()
                     if c.lstrip().startswith("def ") or "db." in c)


def integration_policy(call: ModelCall) -> list:
    text = user_text(call)
    found = []
    if "def fetch_user" in text and "db.get_user(" in text:
        found.append(F("X1", "app/routes.py", 24, "P1", "broken-contract",
                       "routes.py calls db.get_user, but db.py renamed it to fetch_user."))
    return [say_json(FileFindings(findings=found))]


def reviewer_policy(call: ModelCall) -> list:
    """A stand-in for an independent model: confirm only what the cited diff line supports."""
    text = user_text(call)
    findings = json.loads(re.search(r"<findings>\n(.*)\n</findings>", text, re.S).group(1))
    out = []
    for f in findings:
        line_ok = f["quote"] and f["quote"] in PR_FILES.get(f["path"], {}).get(f["line"], "")
        hashed = f["rule"] == "plaintext-password" and "check_hash" in f["quote"]
        decision = "confirmed" if line_ok and not hashed else "rejected"
        out.append(Adjudication(finding_id=f["id"], decision=decision,
                                rationale="cited line supports it" if decision == "confirmed" else "the diff contradicts it"))
    return [say_json(ReviewerReport(adjudications=out, missed_issues=[]))]

In [ ]:
file_worker = Agent(name="File reviewer", model=policy_model(file_worker_policy), output_type=FileFindings,
                    instructions="Review ONE file. Report only issues visible inside it, with line, severity P0-P3, rule, quote.")
integrator = Agent(name="Integration reviewer", model=policy_model(integration_policy), output_type=FileFindings,
                   instructions="You get per-file findings and public interfaces. Report ONLY cross-file issues.")
independent_reviewer = Agent(
    name="Independent reviewer", model=policy_model(reviewer_policy), output_type=ReviewerReport,
    instructions="You did not write these findings. For each one, check the cited lines in the diff. "
                 "Confirm only if the diff shows the problem. Reject anything you cannot tie to a line.")

# Pass 1: per file, in parallel
per_file = await asyncio.gather(*(Runner.run(file_worker, render_file(p)) for p in PR_FILES))
local = [f for r in per_file for f in r.final_output_as(FileFindings).findings]
# Pass 2: integration, cross-file only
integration_input = (f"<findings>\n{json.dumps([f.model_dump() for f in local])}\n</findings>\n"
                     f"<interfaces>\n{interfaces()}\n</interfaces>")
cross = (await Runner.run(integrator, integration_input)).final_output_as(FileFindings).findings
# Pass 3: independent reviewer, fresh run: only the diff, the findings and the criteria
all_findings = local + cross
reviewer_input = f"<diff>\n{DIFF}\n</diff>\n<findings>\n{json.dumps([f.model_dump() for f in all_findings])}\n</findings>"
report = (await Runner.run(independent_reviewer, reviewer_input)).final_output_as(ReviewerReport)

for a in report.adjudications:
    f = next(x for x in all_findings if x.id == a.finding_id)
    print(f"{a.decision:<10} {f.severity} {f.path}:{f.line:<3} {f.rule:<20} ({a.rationale})")
print("\nreviewer input items:", len(independent_reviewer.model.first_call.input), "(one user message, no worker history)")

In [ ]:
# The same pattern, productized in Codex. This notebook never runs the codex CLI; these are the commands you would use.
CODEX_REVIEW_SURFACES = {
    "independent reviewer, CLI": "/review   (base branch, uncommitted changes, a commit, or custom instructions)",
    "independent reviewer, IDE / app": "/review   (base branch or uncommitted changes)",
    "reviewer on a different model": 'review_model = "gpt-6.1-sol"   # config.toml; defaults to the session model',
    "review pass as a CI step": "codex review --base main          # or --uncommitted, or --commit <sha>",
    "GitHub PR review (P0/P1 only)": "@codex review                     # comment on the pull request",
    "sectioned parallel review": "Spawn one subagent for security risks, one for test gaps, and one for maintainability. "
                                 "Wait for all three, then summarize the findings by category.",
}
for purpose, how in CODEX_REVIEW_SURFACES.items():
    print(f"{purpose:<36} {how}")

AGENTS_MD_REVIEW_RULES = """\
## Code Review Rules

### Data access

- Flag SQL built with f-strings or string concatenation from request data.
  Safe path: parameterized queries, db.execute(sql, params).
- Flag route handlers that read another user's record without an authorization check.
"""
print("\n" + AGENTS_MD_REVIEW_RULES)

> **Observe:** the integration pass found the one bug no per-file reviewer could see (`get_user` vs `fetch_user`), and the independent reviewer rejected `A2` because the cited line actually calls `check_hash`. The reviewer's input was a single fresh message: no worker conversation, no `previous_response_id`.

> **Try it:** feed the reviewer the per-file worker's full `to_input_list()` instead of the findings JSON. What could go wrong with a real model? (Hint: README 8.9, the paragraph after the reviewer code: a model reviewing its own work in the same context tends to confirm its reasoning.)

## 8.10 Verification contracts, testing, and tracing an orchestration

Every seam (plan → workers → merge → review) gets a **contract** (owner and scope, evidence, acceptance check in code, on-failure rule, reintegration), an **offline test** with `agents.testing.ScriptedModel`, and a **trace** that groups the whole orchestration. Budget the orchestration in code: `max_turns` bounds one run, not the job. [Theory](README.md#810-verification-contracts-testing-and-tracing-an-orchestration)

In [ ]:
@dataclass
class Contract:
    owner_and_scope: str
    evidence: type[BaseModel]
    acceptance: Callable[[BaseModel, str], list[str]]   # (evidence, path) -> list of problems
    on_failure: str
    reintegration: str


def findings_acceptance(evidence: FileFindings, path: str) -> list[str]:
    problems = []
    for f in evidence.findings:
        if f.path != path:
            problems.append(f"{f.id}: path {f.path} is outside this worker's scope ({path})")
        elif f.line not in PR_FILES[path]:
            problems.append(f"{f.id}: line {f.line} is not in the diff of {path}")
        elif not f.quote.strip():                    # "" is a substring of every line: demand real evidence
            problems.append(f"{f.id}: empty quote; quote the code on line {f.line}")
        elif f.quote.strip() not in PR_FILES[path][f.line]:
            problems.append(f"{f.id}: the quote does not match line {f.line}")
    return problems


FILE_REVIEW_CONTRACT = Contract(
    owner_and_scope="one file, local issues only, read-only",
    evidence=FileFindings,
    acceptance=findings_acceptance,
    on_failure="one retry with the validation errors; then mark the file 'unreviewed' and block approval",
    reintegration="integration pass reads validated findings only; duplicates merge on path + line + rule",
)


async def run_under_contract(agent: Agent, path: str, contract: Contract = FILE_REVIEW_CONTRACT) -> tuple[str, list]:
    prompt = render_file(path)
    for attempt in (1, 2):
        evidence = (await Runner.run(agent, prompt)).final_output_as(contract.evidence)
        problems = contract.acceptance(evidence, path)
        if not problems:
            return "reviewed", evidence.findings
        prompt = f"{render_file(path)}\n<validation_errors>\n" + "\n".join(problems) + "\n</validation_errors>"
    return "unreviewed", problems


BAD_CITATION = Finding(id="D1", path="app/db.py", line=52, severity="P1", rule="missing-index",
                       quote="CREATE INDEX", explanation="users.id has no index")


def sloppy_worker_policy(call: ModelCall) -> list:
    if "<validation_errors>" in user_text(call):                     # fixes it when told exactly what is wrong
        return [say_json(FileFindings(findings=[]))]
    return [say_json(FileFindings(findings=[BAD_CITATION]))]


def stubborn_worker_policy(call: ModelCall) -> list:
    return [say_json(FileFindings(findings=[BAD_CITATION]))]


for label, policy in (("sloppy, then fixes", sloppy_worker_policy), ("never fixes", stubborn_worker_policy)):
    status, detail = await run_under_contract(file_worker.clone(model=policy_model(policy)), "app/db.py")
    print(f"{label:<20} -> {status}: {detail}")

NO_EVIDENCE = BAD_CITATION.model_copy(update={"line": 41, "quote": ""})   # a real line, but nothing quoted
print(f"{'empty quote':<20} -> {findings_acceptance(FileFindings(findings=[NO_EVIDENCE]), 'app/db.py')}")

In [ ]:
# Offline tests of orchestration logic: deterministic, free, and they fail if the workflow stops early.
async def test_narrow_plan_is_rejected() -> None:
    narrow = Plan(subtasks=[Subtask(id="1", facet="security", question="...", context="...")])
    scripted_planner = planner.clone(model=ScriptedModel([[assistant_message(narrow.model_dump_json())]]))
    must_not_run = worker.clone(model=ScriptedModel([]))          # any call to it raises UnexpectedModelCall
    try:
        await orchestrate("Assess vendor X", scripted_planner, must_not_run, offline_synth)
    except ValueError as exc:
        assert "cost" in str(exc) and "operations" in str(exc), exc
    else:
        raise AssertionError("the coverage check did not fire")
    finally:
        scripted_planner.model.assert_complete()                    # the planner ran exactly once
    assert must_not_run.model.calls == (), "a worker ran although the plan was rejected"


async def test_cascade_escalates_unknown_article() -> None:
    t1 = tier1.clone(model=ScriptedModel([[assistant_message(Resolution(kb_article="KB-999", steps=[]).model_dump_json())]]))
    t2 = tier2.clone(model=ScriptedModel([[assistant_message(Resolution(kb_article="KB-303", steps=[]).model_dump_json())]]))
    who, res = await cascade("failover stuck", [t1, t2])
    assert who == t2.name and res.kb_article == "KB-303", (who, res)
    t1.model.assert_complete()
    t2.model.assert_complete()


async def test_handoffs_do_not_fan_out() -> None:
    a = Agent(name="a", model=ScriptedModel([[assistant_message("a")]]))
    b = Agent(name="b", model=ScriptedModel([[assistant_message("b")]]))
    coord = Agent(name="c", handoffs=[a, b], model=ScriptedModel([[function_call("transfer_to_a", {}, call_id="1"),
                                                                    function_call("transfer_to_b", {}, call_id="2")]]))
    result = await Runner.run(coord, "go")
    assert result.last_agent.name == "a" and b.model.remaining_steps == 1


for test in (test_narrow_plan_is_rejected, test_cascade_escalates_unknown_article, test_handoffs_do_not_fan_out):
    await test()
    print("PASS", test.__name__)

In [ ]:
class SpanCollector(TracingProcessor):
    """A local trace processor: records what the SDK traces. Real ones ship spans to your observability stack."""

    def __init__(self) -> None:
        self.traces: list[str] = []
        self.spans: list[tuple[str, str]] = []

    def on_trace_start(self, trace) -> None:
        self.traces.append(f"{trace.name} (group_id={trace.group_id})")

    def on_trace_end(self, trace) -> None:
        pass

    def on_span_start(self, span) -> None:
        pass

    def on_span_end(self, span) -> None:
        data = span.span_data.export() or {}
        self.spans.append((span.span_data.type, data.get("name") or ""))

    def shutdown(self) -> None:
        pass

    def force_flush(self) -> None:
        pass


collector = SpanCollector()
if HAVE_REAL_KEY:
    add_trace_processor(collector)       # keep exporting to the dashboard as well
else:
    set_trace_processors([collector])    # replace the exporter: nothing leaves this machine
    set_tracing_disabled(False)

with trace("Vendor assessment", group_id="vendor-acme"):   # planner, 4 workers and the synthesizer: ONE trace
    await orchestrate("Assess vendor Acme", good_planner, offline_worker, offline_synth)

if not HAVE_REAL_KEY:
    set_tracing_disabled(True)
print("traces:", collector.traces)
print("agent spans:", [name for kind, name in collector.spans if kind == "agent"])
print("span types:", {kind: sum(k == kind for k, _ in collector.spans) for kind, _ in collector.spans})

In [ ]:
class BudgetExceeded(RuntimeError):
    pass


@dataclass
class Budget:
    """An orchestration-wide budget. max_turns bounds ONE run; this bounds the job.

    Usage is known only AFTER a run, so a check after the run always overshoots. Instead, reserve the worst
    case BEFORE each run (max_turns requests: one model call per turn; runner-managed retries are opt-in via
    ModelSettings.retry, so reserve for those too if you enable them) and settle the real usage afterwards.
    That makes requests a hard ceiling, also across parallel workers: a run that could cross it never starts.
    Tokens are checked at admission only; cap each call's output with ModelSettings(max_tokens=...) so the
    runs already in flight can overshoot by a bounded amount.
    """
    max_requests: int
    max_tokens: int
    requests: int = 0       # settled
    tokens: int = 0
    reserved: int = 0       # requests held by runs in flight

    def reserve(self, max_turns: int) -> None:
        if self.requests + self.reserved + max_turns > self.max_requests:
            raise BudgetExceeded(f"refused to start a run: {self.requests} used + {self.reserved} reserved + "
                                 f"{max_turns} more could exceed {self.max_requests} requests")
        if self.tokens >= self.max_tokens:
            raise BudgetExceeded(f"refused to start a run: {self.tokens:,} tokens used, ceiling {self.max_tokens:,}")
        self.reserved += max_turns

    async def run(self, agent: Agent, prompt: str, max_turns: int):
        self.reserve(max_turns)                      # admission control: checked BEFORE spending
        try:
            result = await Runner.run(agent, prompt, max_turns=max_turns)
        except Exception:
            self.requests += max_turns               # usage unknown after a failure: count the worst case
            raise
        finally:
            self.reserved -= max_turns
        usage = result.context_wrapper.usage
        self.requests += usage.requests
        self.tokens += usage.total_tokens
        return result


CALL_USAGE = Usage(requests=1, input_tokens=1_200, output_tokens=200, total_tokens=1_400)  # what each scripted call "costs"
budget_worker = worker.clone(model=policy_model(lambda call: [say("ok")], usage=CALL_USAGE))


async def budgeted_fan_out(prompts: list[str], budget: Budget) -> int:
    done = 0
    for prompt in prompts:                         # sequential here to keep the demo readable
        await budget.run(budget_worker, prompt, max_turns=1)   # a tool-less worker needs one turn
        done += 1
    return done


budget = Budget(max_requests=5, max_tokens=50_000)
try:
    await budgeted_fan_out([f"subtask {i}" for i in range(8)], budget)
except BudgetExceeded as exc:
    print("stopped:", exc)
assert budget.requests <= budget.max_requests
print(f"spent {budget.requests} of {budget.max_requests} requests, {budget.tokens:,} tokens: the ceiling was never crossed")

> **Observe:** the contract turned "the worker returned something" into "the worker returned evidence that passed a check", with a defined fallback (`unreviewed`). The tests ran the real runner, handoff logic and output parsing, with no tokens spent, and `assert_complete()` would fail if a step never ran. The whole vendor assessment shows up as one trace with six agent spans. The budget refused the sixth run *before* it started: usage arrives only after a run, so a ceiling you check afterwards is always overshot, and with parallel workers by a lot.

> **Try it:** add `trace_include_sensitive_data=False` via `RunConfig` to the traced run and compare what the spans contain. `orchestrate` takes no `run_config` yet, so first thread a `run_config=RunConfig(trace_include_sensitive_data=False)` argument through `orchestrate` and `fan_out` into each `Runner.run`. Remember that tracing is unavailable under Zero Data Retention.

## Mini-project: a PR review desk

Combine the chapter into one system:

| Requirement | Mechanism | Section |
|---|---|---|
| A planner turns the PR into per-file and integration subtasks | `output_type=ReviewPlan` on `gpt-6.1-sol` | 8.5 |
| Every changed file is covered, or the planner tries again (at most 2 rounds) | Code-side coverage check + capped re-planning | 8.5 |
| File reviews run in parallel, at most 2 at a time | `asyncio.Semaphore` fan-out | 8.4 |
| Each file review meets a contract, or the file is marked `unreviewed` | Acceptance check + one retry | 8.10 |
| Cross-file issues are found | Integration pass on per-file findings + interfaces | 8.9 |
| Duplicates merge on `path` + `line` + `rule`, and every candidate gets a unique id (workers pick ids independently) | `merge_candidates` in code | 8.10 |
| Findings are confirmed by a reviewer that saw none of the workers' reasoning, on a different model | Fresh `Runner.run`, `gpt-6.1-sol` vs `gpt-6-luna` workers | 8.6, 8.9 |
| Only confirmed P0/P1 findings reach the author; an incomplete review can't approve (a file `unreviewed`, a candidate without exactly one adjudication, a `needs_human` decision, or a reported missed issue) | Code-side filter and verdict | 8.9, 8.10 |
| One trace per PR and a request ceiling for the whole job that no run can cross | `trace(..., group_id=...)` + `Budget.run` (reserve before, settle after) | 8.10 |

It runs offline on scripted models. Set `RUN_LIVE = True` at the end to run the same desk on `MODEL` and `gpt-6.1-sol`.

In [ ]:
class ReviewSubtask(BaseModel):
    id: str
    kind: Literal["file", "integration"]
    path: str | None = Field(description="The file to review; null for the integration subtask")
    focus: list[Literal["security", "correctness", "tests"]]


class ReviewPlan(BaseModel):
    subtasks: list[ReviewSubtask] = Field(max_length=12)


def plan_coverage(plan: ReviewPlan, changed: list[str]) -> list[str]:
    problems = [f"missing file subtask: {p}" for p in changed
                if not any(s.kind == "file" and s.path == p for s in plan.subtasks)]
    if not any(s.kind == "integration" for s in plan.subtasks):
        problems.append("missing integration subtask")
    return problems


SEVERITY_ORDER = {"P0": 0, "P1": 1, "P2": 2, "P3": 3}


def merge_candidates(findings: list[Finding]) -> list[Finding]:
    """Reintegration in code: merge duplicates on (path, line, rule), keeping the first id and quote and the most
    severe severity; then make every id unique. Workers choose ids independently, so two of them can both say "A1"."""
    merged: dict[tuple[str, int, str], Finding] = {}
    for f in findings:
        key = (f.path, f.line, f.rule)
        first = merged.get(key)
        if first is None:
            merged[key] = f
        elif SEVERITY_ORDER[f.severity] < SEVERITY_ORDER[first.severity]:
            merged[key] = first.model_copy(update={"severity": f.severity})
    out, used = [], set()
    for f in merged.values():
        fid, n = f.id, 1
        while fid in used:
            n += 1
            fid = f"{f.id}-{n}"
        used.add(fid)
        out.append(f.model_copy(update={"id": fid}))
    return out


def adjudication_problems(candidates: list[Finding], report: ReviewerReport) -> list[str]:
    """Structured Outputs guarantees the report's SHAPE, not that it covers each candidate exactly once."""
    ids = [a.finding_id for a in report.adjudications]
    known = {f.id for f in candidates}
    problems = [f"unknown finding id: {i}" for i in sorted(set(ids) - known)]
    problems += [f"adjudicated more than once: {i}" for i in sorted({i for i in ids if ids.count(i) > 1})]
    problems += [f"not adjudicated: {i}" for i in sorted(known - set(ids))]
    problems += [f"needs a human: {a.finding_id}" for a in report.adjudications if a.decision == "needs_human"]
    problems += [f"reviewer reported a missed issue: {m}" for m in report.missed_issues]
    return problems


@dataclass
class DeskReport:
    verdict: str
    confirmed: list[Finding]
    unreviewed: list[str]
    plan_rounds: int
    retries: int
    budget: Budget
    open_questions: list[str] = field(default_factory=list)


def desk_planner_policy(call: ModelCall) -> list:
    text = user_text(call)
    paths = re.findall(r"\+\+\+ (\S+)", text)
    if "<coverage_problems>" not in text:          # round 1: the classic over-narrow plan (skips the tests)
        paths = [p for p in paths if not p.startswith("tests/")]
    subtasks = [ReviewSubtask(id=f"f{i}", kind="file", path=p, focus=["security", "correctness"])
                for i, p in enumerate(paths, 1)]
    subtasks.append(ReviewSubtask(id="x1", kind="integration", path=None, focus=["correctness"]))
    return [say_json(ReviewPlan(subtasks=subtasks))]


def desk_file_policy(call: ModelCall) -> list:
    """Per-file reviewer stand-in. On routes.py it first cites a line outside the diff, then fixes it."""
    text = user_text(call)
    path = re.search(r"\+\+\+ (\S+)", text).group(1)
    if path == "app/routes.py" and "<validation_errors>" not in text:
        wrong = Finding(id="R0", path=path, line=99, severity="P2", rule="rate-limit", quote="", explanation="no rate limit")
        return [say_json(FileFindings(findings=LOCAL_FINDINGS[path] + [wrong]))]
    return [say_json(FileFindings(findings=LOCAL_FINDINGS[path]))]


USAGE = Usage(requests=1, input_tokens=1_500, output_tokens=300, total_tokens=1_800)


def build_desk(live: bool) -> dict[str, Agent]:
    pick = lambda model, policy: model if live else policy_model(policy, usage=USAGE)  # noqa: E731
    return {
        "planner": Agent(name="PR planner", output_type=ReviewPlan, model=pick(STRONG_MODEL, desk_planner_policy),
                         instructions="Plan a code review. One subtask of kind 'file' per changed file (path = the file), "
                                      "plus exactly one 'integration' subtask with path null. If <coverage_problems> "
                                      "are listed, fix them."),
        "file": Agent(name="File reviewer", output_type=FileFindings, model=pick(MODEL, desk_file_policy),
                      instructions="Review ONE file's added lines. Report only issues visible inside this file. "
                                   "Each finding: unique id, path, line (a line number shown in the diff), severity "
                                   "P0 (exploitable/data loss) to P3 (nit), rule, quote (the exact code on that line), "
                                   "explanation. If <validation_errors> are listed, fix exactly those."),
        "integration": Agent(name="Integration reviewer", output_type=FileFindings, model=pick(STRONG_MODEL, integration_policy),
                             instructions="You get validated per-file findings and the public interfaces. Report ONLY "
                                          "cross-file issues (contracts, renamed functions, data flow). Use ids X1, X2..."),
        "reviewer": Agent(name="Independent reviewer", output_type=ReviewerReport, model=pick(STRONG_MODEL, reviewer_policy),
                          instructions="You did not write these findings. For each one, check the cited line in the "
                                       "diff. Confirm only if the diff shows the problem; reject anything the diff "
                                       "contradicts; use needs_human when unsure."),
    }

In [ ]:
async def review_pr(pr_id: str, desk: dict[str, Agent], max_plan_rounds: int = 2, file_limit: int = 2,
                    budget: Budget | None = None) -> DeskReport:
    budget = budget or Budget(max_requests=20, max_tokens=200_000)
    changed = list(PR_FILES)
    retries = 0

    async def run(agent: Agent, prompt: str):
        return await budget.run(agent, prompt, max_turns=3)   # refuses to START a run that could cross the ceiling

    with trace("PR review desk", group_id=pr_id):
        # 1) Plan, with a coverage check and capped re-planning
        prompt = f"<diff>\n{DIFF}\n</diff>"
        for plan_round in range(1, max_plan_rounds + 1):
            plan = (await run(desk["planner"], prompt)).final_output_as(ReviewPlan)
            problems = plan_coverage(plan, changed)
            if not problems:
                break
            prompt = f"<diff>\n{DIFF}\n</diff>\n<coverage_problems>\n" + "\n".join(problems) + "\n</coverage_problems>"
        else:
            return DeskReport(f"blocked: plan incomplete after {max_plan_rounds} rounds", [], changed, plan_round, 0, budget)

        # 2) Per-file reviews in parallel under a cap, each under its contract
        gate = asyncio.Semaphore(file_limit)

        async def review_file(path: str) -> tuple[str, str, list]:
            nonlocal retries
            async with gate:
                prompt = render_file(path)
                for attempt in (1, 2):
                    evidence = (await run(desk["file"], prompt)).final_output_as(FileFindings)
                    errors = findings_acceptance(evidence, path)
                    if not errors:
                        return path, "reviewed", evidence.findings
                    retries += 1
                    prompt = f"{render_file(path)}\n<validation_errors>\n" + "\n".join(errors) + "\n</validation_errors>"
                return path, "unreviewed", errors

        outcomes = await asyncio.gather(*(review_file(s.path) for s in plan.subtasks if s.kind == "file"),
                                        return_exceptions=True)
        local, unreviewed = [], []
        for subtask, outcome in zip([s for s in plan.subtasks if s.kind == "file"], outcomes):
            if isinstance(outcome, BudgetExceeded):
                raise outcome                          # out of budget: queued files never started; stop the job
            if isinstance(outcome, Exception) or outcome[1] == "unreviewed":
                unreviewed.append(subtask.path)
            else:
                local += outcome[2]

        # 3) Integration pass: validated findings + interfaces, cross-file issues only
        integration_input = (f"<findings>\n{json.dumps([f.model_dump() for f in local])}\n</findings>\n"
                             f"<interfaces>\n{interfaces()}\n</interfaces>")
        cross = (await run(desk["integration"], integration_input)).final_output_as(FileFindings).findings

        # 4) Independent reviewer: fresh run, different model, only diff + merged, uniquely-identified findings
        candidates = merge_candidates(local + cross)
        reviewer_input = f"<diff>\n{DIFF}\n</diff>\n<findings>\n{json.dumps([f.model_dump() for f in candidates])}\n</findings>"
        report = (await run(desk["reviewer"], reviewer_input)).final_output_as(ReviewerReport)

    # 5) Code decides what reaches the author, and only a complete review can approve
    open_questions = adjudication_problems(candidates, report)
    confirmed_ids = {a.finding_id for a in report.adjudications if a.decision == "confirmed"}
    confirmed = sorted((f for f in candidates if f.id in confirmed_ids and f.severity in ("P0", "P1")),
                       key=lambda f: (SEVERITY_ORDER[f.severity], f.path, f.line))
    if unreviewed:
        verdict = "blocked: incomplete review"
    elif open_questions:
        verdict = "escalate: human review needed"
    elif confirmed:
        verdict = "request changes"
    else:
        verdict = "approve"
    return DeskReport(verdict, confirmed, unreviewed, plan_round, retries, budget, open_questions)


def print_report(r: DeskReport) -> None:
    print(f"verdict: {r.verdict} | plan rounds: {r.plan_rounds} | contract retries: {r.retries} | "
          f"unreviewed: {r.unreviewed or 'none'} | budget used: {r.budget.requests} requests, {r.budget.tokens:,} tokens")
    for q in r.open_questions:
        print(f"  open: {q}")
    for f in r.confirmed:
        print(f"  {f.severity} {f.path}:{f.line}  [{f.rule}] {f.explanation}")

In [ ]:
desk = build_desk(live=False)
desk_report = await review_pr("pr-42", desk)
print_report(desk_report)

assert desk_report.verdict == "request changes"
assert [f.id for f in desk_report.confirmed] == ["A1", "R1", "X1"], [f.id for f in desk_report.confirmed]
assert desk_report.plan_rounds == 2 and desk_report.retries == 1 and not desk_report.unreviewed
assert not desk_report.open_questions and desk_report.budget.requests <= desk_report.budget.max_requests
print("\nAll desk checks pass: over-narrow plan re-planned, bad citation retried, false positive and P3 filtered.")

In [ ]:
# What happens when the budget is too small, a file can't be reviewed, or the review is incomplete? Fail closed.
small = Budget(max_requests=4, max_tokens=200_000)
try:
    await review_pr("pr-42", build_desk(live=False), budget=small)
except BudgetExceeded as exc:
    print("budget:", exc, f"(spent {small.requests} of {small.max_requests})")
assert small.requests <= small.max_requests

broken = build_desk(live=False)
broken["file"] = broken["file"].clone(model=policy_model(
    lambda call: TimeoutError("reviewer down") if "auth.py" in user_text(call) else desk_file_policy(call), usage=USAGE))
print_report(await review_pr("pr-42", broken))


def unsure_reviewer_policy(call: ModelCall) -> list:
    """Schema-valid but incomplete: every finding 'needs_human', plus an issue nobody reported."""
    findings = json.loads(re.search(r"<findings>\n(.*)\n</findings>", user_text(call), re.S).group(1))
    return [say_json(ReviewerReport(
        adjudications=[Adjudication(finding_id=f["id"], rationale="cannot tell from the diff", decision="needs_human")
                       for f in findings],
        missed_issues=["app/auth.py:13 may allow an authentication bypass"]))]


unsure = build_desk(live=False)
unsure["reviewer"] = unsure["reviewer"].clone(model=policy_model(unsure_reviewer_policy, usage=USAGE))
unsure_report = await review_pr("pr-42", unsure)
print_report(unsure_report)
assert unsure_report.verdict.startswith("escalate")

# Workers pick ids independently: two different findings both called "A1", and one finding reported twice.
clash = [F("A1", "app/auth.py", 11, "P1", "sql-injection", "f-string SQL"),
         F("A1", "app/routes.py", 24, "P1", "missing-authz", "no authz check"),
         F("S7", "app/auth.py", 11, "P0", "sql-injection", "same bug, other worker")]
print("\nmerged:", [(f.id, f.path, f.line, f.severity) for f in merge_candidates(clash)])

In [ ]:
# (live, ~$0.02) the same desk with real models: gpt-6-luna file reviewers, gpt-6.1-sol planner/integration/reviewer.
RUN_LIVE = False
if RUN_LIVE:
    live_report = await review_pr("pr-42-live", build_desk(live=True))
    print_report(live_report)

> **Observe:** every safety property came from code: the coverage check forced a second plan, the contract caught a citation of a line that is not in the diff, the reviewer's rejection of `A2` and the P0/P1 filter shaped what the author sees, and a failed file review blocked approval instead of silently dropping `auth.py`, the file with the P0. A reviewer report that was schema-valid but incomplete (`needs_human` everywhere, a missed issue) escalated instead of approving: Structured Outputs guarantees the shape, not the content. The budget refused to start the first file review that could have crossed four requests. The models only planned, reviewed and judged.

> **Try it:** (1) Add a fifth file to `PR_FILES` and watch the scripted planner cover it. (2) Swap the file fan-out for a manager with `as_tool()` workers and `max_function_tool_concurrency=2`. What do you gain (adaptivity) and lose (the contract check now runs where)? (3) Write the desk's `## Code Review Rules` into an `AGENTS.md` so `@codex review` follows the same rules on GitHub.

## Recap

- Pick the cheapest shape: single call → pipeline → routing → fan-out → orchestrator-workers → evaluator loop. Start with one agent; let evals justify splits.
- Pipelines: one job per agent, `output_type` gates checked by an `if`, every step in one trace.
- Routing: classifier + dispatch table (testable alone), handoffs when the specialist owns the reply, tiering with a code check for escalation.
- Fan-out: `asyncio.gather(..., return_exceptions=True)` behind a semaphore; decide per slot whether a gap blocks. Inside an agent: `parallel_tool_calls` vs `max_function_tool_concurrency`.
- Orchestrator-workers: structured `Plan`, code-side coverage check, explicit context per worker, caps on subtasks, rounds and turns.
- Evaluator loops: structured verdict, fresh evaluator context, specific feedback, a cap with a defined fallback.
- Handoffs honor one per turn and cannot merge; agents-as-tools fan out and merge.
- Hosted multi-agent (Responses, Agents API, Codex) trades per-branch control for less code; never execute `multi_agent_call` items.
- Review in three passes with an independent reviewer; Codex ships it as `/review`, `codex review`, `@codex review`.
- Contracts, scripted tests, one trace and an orchestration-wide budget make it production-ready.

Next: [homework](02_homework.ipynb)